# UK Biobank colocalization with 1000 Genomes LD

In [10]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/code/UKB_fine.R
# ============================================================
# UKB blood-trait GWAS fine-mapping with 1000G (KG) LD reference
#   - SuSiE-RSS with UKB-provided LD ("in-sample" panel)
#   - SuSiE-RSS with 1000G-genotype LD (external panel) + kriging QC
#   - base / EB (haochen build) / EB-mix (xuewei build)
#
# Usage: Rscript UKB_fine.R <IDX>
#   IDX is the SLURM array task id; each task processes 30 regions.
# ============================================================

# ---- batch index from command line ----
args <- commandArgs(trailingOnly = TRUE)
IDX  <- as.integer(args[1])
stopifnot(!is.na(IDX), IDX >= 1)

# ---- libraries ----
library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

# ============================================================
# Two susieR builds, each ISOLATED in its own environment so their
# identically-named functions (susie_rss + helpers) never clobber.
# Extracting $susie_rss keeps the function bound to its build's env,
# so its internal helper calls still resolve within that build.
# ============================================================
source_build <- function(dir) {
  rdir <- if (dir.exists(file.path(dir, "R"))) file.path(dir, "R") else dir
  env  <- new.env(parent = globalenv())
  for (f in list.files(rdir, full.names = TRUE, pattern = "\\.R$"))
    sys.source(f, envir = env)
  env
}

susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
susie_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/susieR_July17_2026_singleL/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

sketch_harmonize <- function(target_data, ref_variants, geno_1kg) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(geno_1kg))
  if (anyNA(idx)) stop("Some qced variant_id values not found in 1000G columns.")
  geno2 <- geno_1kg[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    geno2[, flip_index] <- -geno2[, flip_index, drop = FALSE]
  colnames(geno2) <- qced$variants_id_original
  list(sumstat = qced, geno = geno2,
       flipped_prop = length(flip_index) / ncol(geno2))
}


# max inter-CS correlation (NA when < 2 CS), against the LD the fit used
cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- config ----
sumstat_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/UKB_blood_real_data/"
snp_info_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/snp_info/"
out_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/result"
ld_1kg_dir   <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_EUR_hg38"   # BED/BIM/FAM per chr

maf.cut      <- 0.05
missing.cut  <- 0.1
n_ukb_ld_ref <- 10000   # nominal in-sample LD ref size; large => EB ~ off
B_insample  <- 1e6      # in-sample LD is exact -> huge R_finite (EB captures only residual)
n_kg         <- 489     # 1000G EUR sample size for R_finite on KG LD

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

files <- list.files(sumstat_dir, pattern = ".summary_stats.rds", full.names = TRUE)

idx_start <- 15 * (IDX - 1) + 1
idx_end   <- min(15 * IDX, length(files))

message(sprintf("[IDX=%d] processing regions %d-%d of %d",
                IDX, idx_start, idx_end, length(files)))

# ============================================================
# Outer loop: regions
# ============================================================
for (i in idx_start:idx_end) {

  tryCatch({

    file      <- files[i]
    base_file <- basename(file)

    # ---- 1. parse region from filename ----
    parts     <- str_split(base_file, "\\.")[[1]]
    blood_chr <- parts[1]
    start     <- parts[2]
    end       <- parts[3]
    chrom     <- str_remove(str_split(blood_chr, "_")[[1]][2], "chr")
    region    <- paste0("chr", chrom, ":", start, "-", end)

    # skip-existing
    res_file <- file.path(out_dir, paste0("res_", region, ".rds"))
    if (file.exists(res_file)) next

    message(sprintf("[%s] === region %d/%d: %s ===",
                    format(Sys.time(), "%Y-%m-%d %H:%M:%S"), i, idx_end, region))

    # ---- 2. load sumstats + sufficient stats ----
    suff_stat_file <- str_replace(base_file, "summary", "sufficient")
    sumstat_list   <- readRDS(file)
    suff_stat_list <- readRDS(file.path(sumstat_dir, suff_stat_file))

    if (ncol(sumstat_list$Z) == 0) next

    # dbsnp hg38 lookup (CHR/BP already lifted)
    dpsnp_match <- read_tsv(file.path(snp_info_dir, paste0("chrom_", chrom, "_snp.tsv")),
                            show_col_types = FALSE)

    snp_meta     <- sumstat_list$meta
    target_input <- snp_meta %>%
      left_join(dpsnp_match %>% select(SNP, BP), by = c("ID" = "SNP")) %>%
      select(CHR, BP, ID, REF, ALT, maf) %>%
      mutate(original_ID = paste0(ID, "_", REF)) %>%
      arrange(BP) %>%
      na.omit()

    start <- min(target_input$BP)
    end   <- max(target_input$BP)

    # ---- 3. 1000G LD reference ----
    kg_prefix <- file.path(ld_1kg_dir, paste0("1000G_b38_EUR_chr", chrom))
    kg_geno   <- BEDMatrix(kg_prefix)
    kg_bim    <- read.delim(paste0(kg_prefix, ".bim"), header = FALSE, stringsAsFactors = FALSE)
    colnames(kg_geno) <- paste0("chr", kg_bim$V1, ":", kg_bim$V4, ":", kg_bim$V6, ":", kg_bim$V5)

    # subset to gene region
    kg_gene <- kg_bim %>% mutate(idx = row_number()) %>% filter(V4 >= start & V4 <= end)

    kg_mat_raw <- data.matrix(kg_geno[, kg_gene$idx, drop = FALSE])
    kg_ids     <- paste0("chr", kg_gene$V1, ":", kg_gene$V4, ":", kg_gene$V6, ":", kg_gene$V5)
    colnames(kg_mat_raw) <- kg_ids

    # filter_X: missing imputation, MAF, zero-variance -> then scale
    kg_mat <- filter_X(kg_mat_raw, missing_rate_thresh = missing.cut, maf_thresh = maf.cut)
    kg_mat <- scale(kg_mat, center = TRUE, scale = TRUE)

    # sync kg_ref to variants that survived filter_X
    surviving <- colnames(kg_mat)
    kg_ref <- tibble(
      variant_id = surviving,
      chrom      = kg_gene$V1[match(surviving, kg_ids)],
      pos        = kg_gene$V4[match(surviving, kg_ids)],
      A2         = kg_gene$V6[match(surviving, kg_ids)],
      A1         = kg_gene$V5[match(surviving, kg_ids)]
    ) %>% select(-variant_id)

    # ---- 4. trait loop ----
    res_susie <- list()

    for (tissue_idx in seq_len(ncol(sumstat_list$Z))) {

      tissue_name <- colnames(sumstat_list$Z)[tissue_idx]
      n           <- suff_stat_list$N
      z           <- sumstat_list$Z[, tissue_idx]

      message(sprintf("[%s]   trait: %s",
                      format(Sys.time(), "%Y-%m-%d %H:%M:%S"), tissue_name))

      # build pos-sorted sumstat with z
      snp_meta   <- sumstat_list$meta
      snp_meta$z <- z

      sumstat_input <- snp_meta %>%
        left_join(dpsnp_match %>% select(SNP, BP), by = c("ID" = "SNP")) %>%
        select(CHR, BP, ID, REF, ALT, maf, z) %>%
        mutate(original_ID = paste0(ID, "_", REF)) %>%
        arrange(BP) %>%
        na.omit()

      colnames(sumstat_input) <- c("chrom", "pos", "snp", "A2", "A1", "maf", "z", "original_ID")
      sumstat_input <- sumstat_input %>%
        mutate(var_original_name = paste0(chrom, ":", pos, ":", A2, ":", A1))

      # ---- allele QC against KG reference ----
      harm <- sketch_harmonize(target_data  = sumstat_input,
                               ref_variants = kg_ref,
                               geno_1kg     = kg_mat)

      if (ncol(harm$geno) == 0) {
        message(sprintf("  No variants survive harmonization for trait %s - skipping", tissue_name))
        next
      }

      # LD from 1000G reference panel + kriging QC
      kg_LD          <- crossprod(harm$geno) / (nrow(harm$geno) - 1)
      krig           <- kriging_fix(z = harm$sumstat$z, LD = kg_LD, n = n)
      harm$sumstat$z <- krig$z

      # ---- in-sample z / LD aligned to the harmonized variant set ----
      insample_z  <- sumstat_list$Z[harm$sumstat$original_ID, tissue_idx]
      insample_LD <- sumstat_list$LD[harm$sumstat$original_ID, harm$sumstat$original_ID]

      # ==========================================================
      # Fits: base / EB (susie_hc)  +  EB-mix (susie_xw)
      # ==========================================================

      # ---- in-sample panel ----
      insample_res         <- safe_susie(susie_hc, "insample_res",
                                         z = insample_z, R = insample_LD, n = n)


      insample_eb_mix_res  <- safe_susie(susie_hc, "insample_eb_mix_res",
                                         z = insample_z, R = insample_LD, n = n,
                                         R_finite = B_insample, R_mismatch = "eb_mix")

      # ---- 1000G (KG) panel: kriged z against kg_LD ----
      susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n)

      susie_eb_mix_res         <- safe_susie(susie_hc, "susie_eb_mix_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n,
                                         R_finite = n_kg, R_mismatch = "eb_mix")

      susie_eb_mix_res_new <- safe_susie(
          susie_xw, "susie_eb_mix_res_new",
          z = harm$sumstat$z, R = kg_LD, n = n,
          R_finite = n_kg, R_mismatch = "eb_mix")


      susie_ser_res        <- safe_susie(susie_hc, "susie_ser_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n,
                                         L = 1, max_iter = 1)

      # ---- max inter-CS correlation, one per fit ----
      max_cs_corr <- c(
        insample_res         = cs_maxcor(insample_res,         insample_LD),
        insample_eb_mix_res  = cs_maxcor(insample_eb_mix_res,  insample_LD),
        susie_rss_res        = cs_maxcor(susie_rss_res,        kg_LD),
        susie_eb_mix_res         = cs_maxcor(susie_eb_mix_res,         kg_LD),
        susie_eb_mix_res_new   = cs_maxcor(susie_eb_mix_res_new,   kg_LD),
        susie_ser_res        = cs_maxcor(susie_ser_res,        kg_LD)
      )

      res_susie[[tissue_name]] <- list(
        variant_ids          = colnames(harm$geno),
        # --- fits ---
        insample_res         = insample_res,
        insample_eb_mix_res  = insample_eb_mix_res,
        susie_rss_res        = susie_rss_res,
        susie_eb_mix_res         = susie_eb_mix_res,
        susie_eb_mix_res_new     = susie_eb_mix_res_new,
        susie_ser_res        = susie_ser_res,
        # --- diagnostics / bookkeeping ---
        max_cs_corr          = max_cs_corr,
        sample_size          = n,
        flipped_prop         = harm$flipped_prop,
        original_file_path = file,
        kriging_outlier_number = krig$outlier_number,  
        region = region
      )
    }

    # ---- save per-region ----
    if (length(res_susie) == 0) {
      saveRDS("no traits processed",
              file.path(out_dir, paste0("nores_", region, ".rds")))
    } else {
      saveRDS(res_susie, res_file)
    }

  }, error = function(e) {
    cat(sprintf("[FAIL] region i=%d (%s): %s\n",
                i, basename(files[i]), conditionMessage(e)))
  })
}

message(sprintf("[IDX=%d] done.", IDX))

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/code/UKB_fine.R


In [9]:
%%writefile  /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/code/submit_UKB_fine.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J "UKB_ske[1-65]"
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/log/UKB_ske.%J.%I.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/log/UKB_ske.%J.%I.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/
mkdir -p ${workdir}/log
code_dir="$workdir/code"
tmp_root="$workdir/tmp"

tmp_dir="$tmp_root/${LSB_JOBID}_${LSB_JOBINDEX}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

Rscript "$code_dir/UKB_fine.R" "$LSB_JOBINDEX"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/code/submit_UKB_fine.sh


## Coloc

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome//"
job_name="coloc"
job_dir="${base_dir}/${job_name}"

# Source: where the per-gene SuSiE result RDS files live
src_dir="${base_dir}/result"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=25000]
#BSUB -W 36:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/18.Fig5B_5D.UKB_Coloc.ipynb" coloc \
    --in-dir SRC_DIR \
    --out-dir WORK_DIR/result/ \
    --mem 40G -J 50
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|SRC_DIR|${src_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"